# Inventory EDA

Explore the SQLite metadata produced by:

```powershell
uv run -m uni_rag_agent inventory run
```

This notebook reads `data/uni_rag.sqlite` in read-only mode. It does not mutate `Courses`, execute course files, load unsafe artifacts, or write analysis outputs back to the application database.

Use it after an inventory run to understand course-level volume, extension mix, metadata-only files, extraction backlog, failed/skipped rows, and inventory freshness before implementing or tuning extraction/indexing.

## Maintenance Contract

Keep this notebook aligned with Feature 03 inventory output. Update it in the same change whenever `inventory run`, `inventory summary`, the `courses` table, the `files` table, inventory rows in `extraction_runs`, status values, metadata-only reasons, missing-file semantics, or extraction-backlog interpretation changes.

Before committing, clear notebook outputs and execution counts. This notebook should stay read-only: it may inspect generated app data, but it must not write to SQLite, mutate `Courses`, execute course files, or execute course notebooks.

## Setup

This notebook uses pandas DataFrames for tabular analysis. It locates the repository root by walking upward from the current working directory until it finds `pyproject.toml` and `context/`, which makes it work whether Jupyter starts in the repo root or in `notebooks/`.

In [ ]:
from __future__ import annotations

import json
import sqlite3
from pathlib import Path

import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:  # Allows the helper cells to run outside Jupyter for smoke checks.
    Markdown = None
    display = print


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "context").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from this project or notebooks/.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = REPO_ROOT / "data" / "uni_rag.sqlite"
COURSES_ROOT = REPO_ROOT / "Courses"

REQUIRED_TABLES = [
    "courses",
    "files",
    "extraction_runs",
    "extracted_documents",
    "chunks",
    "chunk_fts",
    "embeddings",
    "data_summaries",
    "search_runs",
    "search_results",
    "evidence_packets",
    "answers",
]


def display_md(text: str) -> None:
    if Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def fmt_bytes(value: int | float | None) -> str:
    if value is None:
        return ""
    size = float(value)
    units = ["B", "KB", "MB", "GB", "TB"]
    unit_index = 0
    while abs(size) >= 1024 and unit_index < len(units) - 1:
        size /= 1024
        unit_index += 1
    if unit_index == 0:
        return f"{int(size):,} {units[unit_index]}"
    return f"{size:,.2f} {units[unit_index]}"


def display_df(df: pd.DataFrame, max_rows: int = 25) -> None:
    if df.empty:
        display_md("_No rows._")
        return
    visible = df.head(max_rows)
    display(visible)
    if len(df) > len(visible):
        display_md(f"_Showing {len(visible):,} of {len(df):,} rows._")


def display_bar_df(df: pd.DataFrame, label_col: str, value_col: str, max_rows: int = 20) -> None:
    if df.empty:
        display_md("_No rows._")
        return
    visible = df.head(max_rows).copy()
    values = pd.to_numeric(visible[value_col], errors="coerce").fillna(0)
    max_value = values.max()
    bar_width = 30
    if max_value > 0:
        visible["bar"] = values.map(
            lambda value: "#" * max(1, round((value / max_value) * bar_width))
            if value > 0
            else ""
        )
    else:
        visible["bar"] = ""
    ordered_columns = [label_col, "bar", value_col] + [
        column for column in visible.columns if column not in {label_col, "bar", value_col}
    ]
    display(visible[ordered_columns])
    if len(df) > len(visible):
        display_md(f"_Showing {len(visible):,} of {len(df):,} rows._")


pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)


display_md(f"**Repo root:** `{REPO_ROOT}`  \n**SQLite database:** `{DB_PATH}`")


## Open The Inventory Database

The connection uses SQLite URI `mode=ro` and `PRAGMA query_only = ON`. If a long-running inventory process still holds the database lock, wait for it to finish before running the analysis cells.

In [ ]:
def connect_inventory_db(db_path: Path) -> sqlite3.Connection:
    if not db_path.is_file():
        raise FileNotFoundError(
            f"Inventory database not found at {db_path}. Run "
            "`uv run -m uni_rag_agent storage init`, then "
            "`uv run -m uni_rag_agent inventory run`."
        )
    uri = db_path.resolve().as_uri() + "?mode=ro"
    connection = sqlite3.connect(uri, uri=True, timeout=5)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    connection.execute("PRAGMA query_only = ON")
    return connection


def query_df(sql: str, params: tuple[object, ...] | list[object] = ()) -> pd.DataFrame:
    try:
        return pd.read_sql_query(sql, conn, params=params)
    except sqlite3.OperationalError as exc:
        display_md(
            f"**SQLite query failed:** `{exc}`\n\n"
            "If this reports `database is locked`, finish or stop the active inventory process and rerun this cell."
        )
        raise


def scalar(sql: str, params: tuple[object, ...] = ()) -> object:
    df = query_df(sql, params)
    if df.empty:
        return None
    return df.iat[0, 0]


conn = connect_inventory_db(DB_PATH)
display_md("Connected in read-only mode.")


## Database Health And Latest Runs

In [ ]:
present_tables = {
    row["name"]
    for row in query_df(
        """
        SELECT name
        FROM sqlite_master
        WHERE type IN ('table', 'view')
        """
    ).to_dict("records")
}

table_health = []
for table_name in REQUIRED_TABLES:
    if table_name in present_tables:
        try:
            row_count = scalar(f"SELECT COUNT(*) FROM {table_name}")
        except sqlite3.OperationalError as exc:
            row_count = f"error: {exc}"
    else:
        row_count = None
    table_health.append(
        {"table": table_name, "present": table_name in present_tables, "row_count": row_count}
    )

display_df(pd.DataFrame(table_health), max_rows=50)

run_rows = query_df(
    """
    SELECT id, started_at, finished_at, status, files_seen, files_indexed,
           files_metadata_only, files_failed, error, config_json
    FROM extraction_runs
    ORDER BY id DESC
    LIMIT 10
    """
)
def run_type_from_config(config_json: object) -> str:
    try:
        return json.loads(config_json or "{}").get("run_type", "extraction")
    except json.JSONDecodeError:
        return "unknown"


if not run_rows.empty:
    run_rows["run_type"] = run_rows["config_json"].map(run_type_from_config)
    run_rows = run_rows.drop(columns=["config_json"])
display_md("### Latest `extraction_runs` Rows")
display_df(run_rows, max_rows=10)


## Overall Inventory Metrics

In [ ]:
metrics = {
    "courses": scalar("SELECT COUNT(*) FROM courses"),
    "files": scalar("SELECT COUNT(*) FROM files"),
    "bytes": scalar("SELECT COALESCE(SUM(size_bytes), 0) FROM files"),
    "pending_files": scalar("SELECT COUNT(*) FROM files WHERE index_status = 'pending'"),
    "metadata_only_files": scalar("SELECT COUNT(*) FROM files WHERE index_status = 'metadata_only'"),
    "failed_files": scalar("SELECT COUNT(*) FROM files WHERE index_status = 'failed'"),
    "skipped_files": scalar("SELECT COUNT(*) FROM files WHERE index_status = 'skipped'"),
}
metrics_df = pd.DataFrame(
    [
        {"metric": key, "value": fmt_bytes(value) if key == "bytes" else value}
        for key, value in metrics.items()
    ]
)
display_df(metrics_df, max_rows=20)


## Course-Level Distribution

Use this to find courses that dominate the archive by file count or storage size, and to identify where extraction backlog is concentrated.

In [ ]:
course_rows = query_df(
    """
    SELECT
        COALESCE(c.name, '<Courses root>') AS course,
        COUNT(f.id) AS file_count,
        COALESCE(SUM(f.size_bytes), 0) AS total_bytes,
        ROUND(COALESCE(SUM(f.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_mb,
        SUM(CASE WHEN f.index_status = 'pending' THEN 1 ELSE 0 END) AS pending_files,
        SUM(CASE WHEN f.index_status = 'metadata_only' THEN 1 ELSE 0 END) AS metadata_only_files,
        SUM(CASE WHEN f.index_status = 'failed' THEN 1 ELSE 0 END) AS failed_files,
        SUM(CASE WHEN f.index_status = 'skipped' THEN 1 ELSE 0 END) AS skipped_files
    FROM files AS f
    LEFT JOIN courses AS c ON c.id = f.course_id
    GROUP BY COALESCE(c.name, '<Courses root>')
    ORDER BY file_count DESC, total_bytes DESC
    """
)
display_bar_df(course_rows, "course", "file_count", max_rows=20)
display_df(course_rows, max_rows=30)


## Categories And Statuses

In [ ]:
category_rows = query_df(
    """
    SELECT
        category,
        COUNT(*) AS files,
        ROUND(100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM files), 0), 2) AS pct_files,
        COALESCE(SUM(size_bytes), 0) AS total_bytes,
        ROUND(COALESCE(SUM(size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_mb
    FROM files
    GROUP BY category
    ORDER BY files DESC, total_bytes DESC
    """
)
display_bar_df(category_rows, "category", "files", max_rows=20)
display_df(category_rows, max_rows=30)

status_by_category = query_df(
    """
    SELECT category, index_status, COUNT(*) AS files
    FROM files
    GROUP BY category, index_status
    ORDER BY category, index_status
    """
)
display_md("### Status By Category")
display_df(status_by_category, max_rows=80)


## Extension Analysis

In [17]:
extension_rows = query_df(
    """
    SELECT
        COALESCE(NULLIF(extension, ''), '<none>') AS extension,
        COUNT(*) AS files,
        ROUND(100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM files), 0), 2) AS pct_files,
        COALESCE(SUM(size_bytes), 0) AS total_bytes,
        ROUND(COALESCE(SUM(size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_mb,
        MAX(size_bytes) AS largest_file_bytes
    FROM files
    GROUP BY COALESCE(NULLIF(extension, ''), '<none>')
    ORDER BY files DESC, total_bytes DESC
    LIMIT 50
    """
)
display_bar_df(extension_rows, "extension", "files", max_rows=25)
display_df(extension_rows, max_rows=50)

largest_by_extension = query_df(
    """
    SELECT
        COALESCE(NULLIF(extension, ''), '<none>') AS extension,
        relative_path,
        category,
        index_status,
        size_bytes,
        ROUND(size_bytes / 1024.0 / 1024.0, 2) AS size_mb
    FROM files AS f
    WHERE size_bytes = (
        SELECT MAX(f2.size_bytes)
        FROM files AS f2
        WHERE COALESCE(NULLIF(f2.extension, ''), '<none>') = COALESCE(NULLIF(f.extension, ''), '<none>')
    )
    ORDER BY size_bytes DESC
    LIMIT 25
    """
)
display_md("### Largest File Per Extension")
display_df(largest_by_extension, max_rows=25)


### Largest File Per Extension

,extension,relative_path,category,index_status,size_bytes,size_mb
0,.bin,NLP\Assignments\GoogleNews-vectors-negative300.bin,model_metadata_only,metadata_only,3644258522,3475.44
1,.joblib,Graduation Project\Body Comp\models\checkpoints\rf\model_rf.joblib,model_metadata_only,metadata_only,2561201961,2442.55
2,.joblib,Graduation Project\Full\backend\app\bodycomp\models\checkpoints\rf\model_rf.joblib,model_metadata_only,metadata_only,2561201961,2442.55
3,.cab,Database\Lab\Lab 0\OracleXE213_Win64\DB.cab,installer_metadata_only,metadata_only,1953428809,1862.93
4,.zip,NLP\Assignments\GoogleNews-vectors-negative300.bin.zip,archive_metadata_only,metadata_only,1760926034,1679.35
5,.7z,Object Oriented Programming\Lab\Lab 1 2 3 4 5 6 7 8 Recordings .h & .cpp files.7z,archive_metadata_only,metadata_only,1376140188,1312.39
6,.mp4,Computer and Society\Project\20251218_224104.mp4,media_metadata_only,metadata_only,665869591,635.02
7,.mov,Data Structures and Intro to Algorithms\Past Papers\ds review\fall final 21.mov,media_metadata_only,metadata_only,547709495,522.34
8,.weights,Pattern Recognition\Assignments\Ass 6\yolov3.weights,model_metadata_only,metadata_only,248007048,236.52
9,.rar,Operating Systems\Lab\Lab 10\Socket and Threading Recordings.rar,archive_metadata_only,metadata_only,206320530,196.76


## Metadata-Only Reasons

These counts explain why files are intentionally excluded from extraction/indexing.

In [18]:
reason_rows = query_df(
    """
    SELECT
        reason_not_indexed,
        COUNT(*) AS files,
        ROUND(100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM files), 0), 2) AS pct_files,
        COALESCE(SUM(size_bytes), 0) AS total_bytes,
        ROUND(COALESCE(SUM(size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_mb
    FROM files
    WHERE reason_not_indexed IS NOT NULL AND reason_not_indexed <> ''
    GROUP BY reason_not_indexed
    ORDER BY files DESC, total_bytes DESC
    """
)
display_bar_df(reason_rows, "reason_not_indexed", "files", max_rows=20)
display_df(reason_rows, max_rows=50)


,reason_not_indexed,bar,files,pct_files,total_bytes,total_mb
0,standalone image metadata-only by project decision,##############################,26615,78.48,975529166,930.34
1,unknown or unsupported extension metadata-only,#,509,1.50,210543854,200.79
2,audio/video media metadata-only; transcription is opt-in later,#,70,0.21,9233436476,8805.69
3,model or serialized artifact metadata-only; unsafe or noisy for MVP indexing,#,23,0.07,9424016062,8987.44
4,archive metadata-only; archives are not decompressed,#,12,0.04,3501604308,3339.39
5,installer metadata-only; installers are never executed,#,4,0.01,1973861033,1882.42


,reason_not_indexed,files,pct_files,total_bytes,total_mb
0,standalone image metadata-only by project decision,26615,78.48,975529166,930.34
1,unknown or unsupported extension metadata-only,509,1.50,210543854,200.79
2,audio/video media metadata-only; transcription is opt-in later,70,0.21,9233436476,8805.69
3,model or serialized artifact metadata-only; unsafe or noisy for MVP indexing,23,0.07,9424016062,8987.44
4,archive metadata-only; archives are not decompressed,12,0.04,3501604308,3339.39
5,installer metadata-only; installers are never executed,4,0.01,1973861033,1882.42


## Extraction Backlog

`pending` files are the immediate input to Feature 04 extraction and Feature 05 data summaries.

In [19]:
pending_by_category_extension = query_df(
    """
    SELECT
        category,
        COALESCE(NULLIF(extension, ''), '<none>') AS extension,
        COUNT(*) AS pending_files,
        COALESCE(SUM(size_bytes), 0) AS total_bytes,
        ROUND(COALESCE(SUM(size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_mb
    FROM files
    WHERE index_status = 'pending'
    GROUP BY category, COALESCE(NULLIF(extension, ''), '<none>')
    ORDER BY pending_files DESC, total_bytes DESC
    """
)
display_df(pending_by_category_extension, max_rows=50)

pending_by_course = query_df(
    """
    SELECT
        COALESCE(c.name, '<Courses root>') AS course,
        COUNT(*) AS pending_files,
        COALESCE(SUM(f.size_bytes), 0) AS total_bytes,
        ROUND(COALESCE(SUM(f.size_bytes), 0) / 1024.0 / 1024.0, 2) AS total_mb
    FROM files AS f
    LEFT JOIN courses AS c ON c.id = f.course_id
    WHERE f.index_status = 'pending'
    GROUP BY COALESCE(c.name, '<Courses root>')
    ORDER BY pending_files DESC, total_bytes DESC
    LIMIT 30
    """
)
display_md("### Pending Files By Course")
display_df(pending_by_course, max_rows=30)

largest_pending = query_df(
    """
    SELECT
        COALESCE(c.name, '<Courses root>') AS course,
        f.relative_path,
        f.extension,
        f.category,
        f.size_bytes,
        ROUND(f.size_bytes / 1024.0 / 1024.0, 2) AS size_mb,
        f.modified_at
    FROM files AS f
    LEFT JOIN courses AS c ON c.id = f.course_id
    WHERE f.index_status = 'pending'
    ORDER BY f.size_bytes DESC
    LIMIT 25
    """
)
display_md("### Largest Pending Files")
display_df(largest_pending, max_rows=25)


,category,extension,pending_files,total_bytes,total_mb
0,data_schema,.json,5131,268978060,256.52
1,document,.pdf,436,1218659443,1162.20
2,document,.txt,381,329065,0.31
3,notebook,.ipynb,210,129639993,123.63
4,code,.py,133,748149,0.71
5,document,.docx,113,14796395,14.11
6,slides,.pptx,91,388949738,370.93
7,data_schema,.csv,48,57877455,55.20
8,code,.r,45,203403,0.19
9,code,.cpp,21,40482,0.04


### Pending Files By Course

,course,pending_files,total_bytes,total_mb
0,Graduation Project,5582,276396432,263.59
1,Database,96,37030625,35.32
2,Data Eng,84,36318799,34.64
3,Stats for Data Science,75,89573204,85.42
4,Algorithms Design and Analysis,72,181988457,173.56
5,Artificial Intelligence,62,26608825,25.38
6,Pattern Recognition,59,64380077,61.40
7,Operating Systems,53,68118000,64.96
8,Machine Learning,40,111339671,106.18
9,Data Structures and Intro to Algorithms,40,92602365,88.31


### Largest Pending Files

,course,relative_path,extension,category,size_bytes,size_mb,modified_at
0,Uni Textbooks,"Uni Textbooks\Passages Level 2 Students Book (Jack C. Richards, Chuck Sandy) (z-lib.org).pdf",.pdf,document,76622216,73.07,2022-10-12T13:46:22+00:00
1,Chinese,Chinese\20240103 Chinese Characters Animation.pptx,.pptx,slides,67997719,64.85,2024-02-01T04:37:27+00:00
2,NLP,NLP\9- Transformer and Generative models.pptx,.pptx,slides,50432443,48.10,2026-01-19T15:29:15.062000+00:00
3,DIP,DIP\Archive Notes.pdf,.pdf,document,45227284,43.13,2026-01-18T12:16:03.130000+00:00
4,Stats for Data Science,"Stats for Data Science\Books\Introduction to Probability and Statistics - Metric Version, 15e (William Mendenhall, Barbara M. Beaver etc...",.pdf,document,41194120,39.29,2024-04-07T19:05:51+00:00
5,Uni Textbooks,Uni Textbooks\rosen_discrete_mathematics_and_its_applications_7th_edition.pdf,.pdf,document,37963820,36.21,2023-03-08T11:32:12+00:00
6,Chinese,Chinese\20231220 Chinese Characters.pptx,.pptx,slides,35553492,33.91,2024-01-31T20:14:12+00:00
7,NLP,NLP\3- Word Vectors and Embeddings.pptx,.pptx,slides,35499156,33.85,2025-11-11T17:57:02.991000+00:00
8,Machine Learning,Machine Learning\TextBook-Aurélien-Géron-Hands-On-Machine-Learning-with-Scikit-Learn-Keras-and-Tensorflow_-Concepts-Tools-and-Techniques...,.pdf,document,33053848,31.52,2025-06-02T23:37:02+00:00
9,Data Structures and Intro to Algorithms,Data Structures and Intro to Algorithms\Slides and Notes\pq.pdf,.pdf,document,32277244,30.78,2024-03-19T13:17:43+00:00


## Failed Or Skipped Rows

Use this section to check permission errors, hash failures, files missing from the latest inventory run, or other diagnostics that should be handled before extraction.

In [20]:
failed_skipped_summary = query_df(
    """
    SELECT index_status, reason_not_indexed, COUNT(*) AS files
    FROM files
    WHERE index_status IN ('failed', 'skipped')
    GROUP BY index_status, reason_not_indexed
    ORDER BY files DESC
    """
)
display_df(failed_skipped_summary, max_rows=50)

failed_skipped_samples = query_df(
    """
    SELECT
        COALESCE(c.name, '<Courses root>') AS course,
        f.relative_path,
        f.extension,
        f.category,
        f.index_status,
        f.reason_not_indexed,
        f.size_bytes,
        f.last_seen_at
    FROM files AS f
    LEFT JOIN courses AS c ON c.id = f.course_id
    WHERE f.index_status IN ('failed', 'skipped')
    ORDER BY f.index_status, f.size_bytes DESC
    LIMIT 50
    """
)
display_md("### Failed/Skipped Samples")
display_df(failed_skipped_samples, max_rows=50)


_No rows._

### Failed/Skipped Samples

_No rows._

## Inventory Freshness

In [21]:
freshness = query_df(
    """
    SELECT
        MIN(discovered_at) AS first_discovered_at,
        MAX(discovered_at) AS last_discovered_at,
        MIN(last_seen_at) AS oldest_last_seen_at,
        MAX(last_seen_at) AS newest_last_seen_at,
        COUNT(DISTINCT last_seen_at) AS distinct_last_seen_timestamps
    FROM files
    """
)
display_df(freshness, max_rows=5)

run_history = query_df(
    """
    SELECT id, started_at, finished_at, status, files_seen, files_metadata_only, files_failed, error
    FROM extraction_runs
    ORDER BY id DESC
    LIMIT 20
    """
)
display_md("### Run History")
display_df(run_history, max_rows=20)


,first_discovered_at,last_discovered_at,oldest_last_seen_at,newest_last_seen_at,distinct_last_seen_timestamps
0,2026-06-25T21:22:38.461169+00:00,2026-06-25T21:22:38.461169+00:00,2026-06-25T21:22:38.461169+00:00,2026-06-25T21:22:38.461169+00:00,1


### Run History

,id,started_at,finished_at,status,files_seen,files_metadata_only,files_failed,error
0,1,2026-06-25T21:22:38.461169+00:00,2026-06-25T21:42:58.532961+00:00,completed,33912,27233,0,None


## Optional File Deep Dive

Set one or more filters, then run the cell to inspect matching file rows. This reads metadata only.

In [22]:
COURSE_FILTER = None       # Example: "Information Retrieval"
EXTENSION_FILTER = None    # Example: ".ipynb"
CATEGORY_FILTER = None     # Example: "data_schema"
STATUS_FILTER = "pending"  # Example: "metadata_only", "failed", or None
LIMIT = 100

where = []
params: list[object] = []
if COURSE_FILTER:
    where.append("COALESCE(c.name, '<Courses root>') = ?")
    params.append(COURSE_FILTER)
if EXTENSION_FILTER:
    where.append("f.extension = ?")
    params.append(EXTENSION_FILTER)
if CATEGORY_FILTER:
    where.append("f.category = ?")
    params.append(CATEGORY_FILTER)
if STATUS_FILTER:
    where.append("f.index_status = ?")
    params.append(STATUS_FILTER)

sql = """
SELECT
    COALESCE(c.name, '<Courses root>') AS course,
    f.relative_path,
    f.filename,
    f.extension,
    f.category,
    f.index_status,
    f.reason_not_indexed,
    f.size_bytes,
    ROUND(f.size_bytes / 1024.0 / 1024.0, 2) AS size_mb,
    f.modified_at,
    f.last_seen_at
FROM files AS f
LEFT JOIN courses AS c ON c.id = f.course_id
"""
if where:
    sql += " WHERE " + " AND ".join(where)
sql += " ORDER BY f.size_bytes DESC, f.relative_path LIMIT ?"
params.append(LIMIT)

display_df(query_df(sql, params), max_rows=LIMIT)


,course,relative_path,filename,extension,category,index_status,reason_not_indexed,size_bytes,size_mb,modified_at,last_seen_at
0,Uni Textbooks,"Uni Textbooks\Passages Level 2 Students Book (Jack C. Richards, Chuck Sandy) (z-lib.org).pdf","Passages Level 2 Students Book (Jack C. Richards, Chuck Sandy) (z-lib.org).pdf",.pdf,document,pending,None,76622216,73.07,2022-10-12T13:46:22+00:00,2026-06-25T21:22:38.461169+00:00
1,Chinese,Chinese\20240103 Chinese Characters Animation.pptx,20240103 Chinese Characters Animation.pptx,.pptx,slides,pending,None,67997719,64.85,2024-02-01T04:37:27+00:00,2026-06-25T21:22:38.461169+00:00
2,NLP,NLP\9- Transformer and Generative models.pptx,9- Transformer and Generative models.pptx,.pptx,slides,pending,None,50432443,48.10,2026-01-19T15:29:15.062000+00:00,2026-06-25T21:22:38.461169+00:00
3,DIP,DIP\Archive Notes.pdf,Archive Notes.pdf,.pdf,document,pending,None,45227284,43.13,2026-01-18T12:16:03.130000+00:00,2026-06-25T21:22:38.461169+00:00
4,Stats for Data Science,"Stats for Data Science\Books\Introduction to Probability and Statistics - Metric Version, 15e (William Mendenhall, Barbara M. Beaver etc...","Introduction to Probability and Statistics - Metric Version, 15e (William Mendenhall, Barbara M. Beaver etc.) (Z-Library).pdf",.pdf,document,pending,None,41194120,39.29,2024-04-07T19:05:51+00:00,2026-06-25T21:22:38.461169+00:00
...,...,...,...,...,...,...,...,...,...,...,...
95,Business Intelligence,Business Intelligence\13_EBA3eSolutionsChapter13.pdf,13_EBA3eSolutionsChapter13.pdf,.pdf,document,pending,None,3844754,3.67,2025-11-28T16:27:09.093000+00:00,2026-06-25T21:22:38.461169+00:00
96,Artificial Intelligence,Artificial Intelligence\Ch3 - Solving Problems by Searching.pptx.pdf,Ch3 - Solving Problems by Searching.pptx.pdf,.pdf,document,pending,None,3828068,3.65,2024-11-24T21:52:51+00:00,2026-06-25T21:22:38.461169+00:00
97,Business Intelligence,Business Intelligence\2. Linear Programming.pdf,2. Linear Programming.pdf,.pdf,document,pending,None,3787067,3.61,2025-11-28T15:10:16.012000+00:00,2026-06-25T21:22:38.461169+00:00
98,Database,Database\Chapter06.ppt,Chapter06.ppt,.ppt,slides,pending,None,3718144,3.55,2025-06-01T12:28:08+00:00,2026-06-25T21:22:38.461169+00:00


## What To Look For

- Courses with unusually high file counts or total bytes may need targeted extraction smoke tests.
- Large `pending` files should be checked before extraction because they can dominate runtime.
- High metadata-only counts are expected for images, media, archives, installers, binaries, and model artifacts.
- `failed` or `skipped` rows should be reviewed before trusting extraction coverage.
- Extension/category mixes help decide the next extractor priority after Feature 04 starts.

In [23]:
# Run when finished if you want to close the read-only handle explicitly.
# conn.close()
